# 🧪 Inference — Without Pipeline

We trained the model. Now let's use it to predict a **new student's** score.  

Sounds simple... but watch what we actually have to do. 👀

In [1]:
import pandas as pd
import numpy as np
import joblib

## Step 1 — Load the Saved Model

We also need to load the scaler and column list we saved separately.

In [2]:
model           = joblib.load("without_pipeline.pkl")
scaler          = joblib.load("scaler.pkl")
training_columns = joblib.load("training_columns.pkl")

print("Loaded model, scaler, and column list.")

Loaded model, scaler, and column list.


## Step 2 — Create a New Student Sample

In [3]:
new_student = pd.DataFrame([{
    "study_hours"          : 6.0,
    "attendance_percentage": 80.0,
    "sleep_hours"          : 7.0,
    "previous_exam_score"  : 70,
    "internet_access"      : "Yes",
    "parental_education"   : "Graduate",
    "extracurricular_activity": "Yes",
    "part_time_job"        : "No",
    "motivation_level"     : "Medium"
}])

new_student

,study_hours,attendance_percentage,sleep_hours,previous_exam_score,internet_access,parental_education,extracurricular_activity,part_time_job,motivation_level
0,6.0,80.0,7.0,70,Yes,Graduate,Yes,No,Medium


## Step 3 — Reapply Preprocessing Manually

> **We need to remember the SAME preprocessing used during training.**  
> Same columns. Same order. Same scaler.

In [4]:
numerical_cols  = ["study_hours", "attendance_percentage", "sleep_hours", "previous_exam_score"]
categorical_cols = ["internet_access", "parental_education", "extracurricular_activity", "part_time_job", "motivation_level"]

In [5]:
# One-hot encode — same categories as training
new_encoded = pd.get_dummies(new_student, columns=categorical_cols)

print("Columns after encoding:", new_encoded.columns.tolist())

Columns after encoding: ['study_hours', 'attendance_percentage', 'sleep_hours', 'previous_exam_score', 'internet_access_Yes', 'parental_education_Graduate', 'extracurricular_activity_Yes', 'part_time_job_No', 'motivation_level_Medium']


In [6]:
# ⚠️  Column order must match training data exactly
# Add any columns that exist in training but not in new_encoded
for col in training_columns:
    if col not in new_encoded.columns:
        new_encoded[col] = 0

# Drop extra columns not seen during training
new_encoded = new_encoded[training_columns]

print("Columns aligned. Count:", new_encoded.shape[1])

Columns aligned. Count: 20


In [7]:
# ⚠️  Apply the SAME scaler that was used during training
new_encoded[numerical_cols] = scaler.transform(new_encoded[numerical_cols])

new_encoded

,study_hours,attendance_percentage,sleep_hours,previous_exam_score,internet_access_No,internet_access_Yes,parental_education_Graduate,parental_education_High School,parental_education_HighSchool,parental_education_Highschool,parental_education_Postgraduate,parental_education_Primary,extracurricular_activity_No,extracurricular_activity_Yes,part_time_job_No,part_time_job_Yes,motivation_level_High,motivation_level_Low,motivation_level_Medim,motivation_level_Medium
0,0.335585,0.13078,0.152252,0.163962,0,True,True,0,0,0,0,0,0,True,True,0,0,0,0,True


## Step 4 — Make Prediction

In [8]:
prediction = model.predict(new_encoded)
print(f"Predicted Final Score: {prediction[0]:.2f}")

Predicted Final Score: 70.85


---

### 😤 That was a lot of work just to predict ONE new student.

Here's what we had to remember:
- Which columns are numerical vs categorical
- Which scaler was used (and load it separately)
- Which columns were created after encoding
- The exact column order from training
- How to align columns before predicting

**If we forget any of these steps, the model breaks silently or gives wrong results.**

> 💡 Next: `with_pipeline.ipynb` — let's see a cleaner way.